# Inspect Ingested Data Tables

This notebook helps you inspect the SQLite bronze tables used by Skimmer.

- Default DB path: `data/skimmer.db`
- Override with env var: `SKIMMER_DB_PATH`
- It lists tables, shows schema, row counts, and recent records.

In [1]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

DEFAULT_DB_PATH = Path('data/skimmer.db')
DB_PATH = Path(os.environ.get('SKIMMER_DB_PATH', DEFAULT_DB_PATH)).expanduser()

print(f'Using database: {DB_PATH.resolve()}')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH}')

Using database: /home/orangepi/code_projects/skimmer/data/skimmer.db


In [2]:
connection = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

if tables_df.empty:
    raise RuntimeError('No tables found in the database.')

display(tables_df)

,table_name
0,bronze_socialblade_channel_stats
1,bronze_socialblade_daily_channel_metrics
2,bronze_vidiq_channel_stats
3,bronze_youtube_skimmed


In [3]:
def table_schema(table_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"PRAGMA table_info({table_name})", connection)


def row_count(table_name: str) -> int:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    return int(pd.read_sql_query(query, connection).iloc[0]['row_count'])


def preview_table(table_name: str, limit: int = 25) -> pd.DataFrame:
    query = f"SELECT * FROM {table_name} ORDER BY id DESC LIMIT {int(limit)}"
    df = pd.read_sql_query(query, connection)

    if 'raw_record_json' in df.columns:
        def parse_raw(value):
            if value is None:
                return None
            try:
                return json.loads(value)
            except (json.JSONDecodeError, TypeError):
                return value

        df['raw_record_json_parsed'] = df['raw_record_json'].map(parse_raw)

    return df


In [4]:
summary_rows = []
for table_name in tables_df['table_name'].tolist():
    summary_rows.append({'table_name': table_name, 'row_count': row_count(table_name)})

summary_df = pd.DataFrame(summary_rows).sort_values(['row_count', 'table_name'], ascending=[False, True])
display(summary_df)

,table_name,row_count
3,bronze_youtube_skimmed,1399
1,bronze_socialblade_daily_channel_metrics,14
2,bronze_vidiq_channel_stats,6
0,bronze_socialblade_channel_stats,1


In [5]:
for table_name in tables_df['table_name'].tolist():
    print('\n' + '=' * 100)
    print(f'TABLE: {table_name}')
    print('=' * 100)

    print('\nSchema:')
    display(table_schema(table_name))

    print('\nLatest rows:')
    display(preview_table(table_name, limit=20))


TABLE: bronze_socialblade_channel_stats

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,create_dt,TEXT,1,None,0
2,2,ingested_at,TEXT,1,None,0
3,3,channel_id,TEXT,0,None,0
4,4,subscribers,,0,None,0
5,5,subscribers_change,,0,None,0
6,6,subscribers_change_percentage,,0,None,0
7,7,views_change,,0,None,0
8,8,views_change_percentage,,0,None,0
9,9,raw_record_json,TEXT,1,None,0



Latest rows:


,id,create_dt,ingested_at,channel_id,subscribers,subscribers_change,subscribers_change_percentage,views_change,views_change_percentage,raw_record_json,raw_record_json_parsed
0,1,2026-07-18T05:19:34.150502+00:00,2026-07-18T05:19:34.150502+00:00,@1hundred1,153000,None,None,None,None,"{""channel_id"": ""@1hundred1"", ""subscribers"": 15...","{'channel_id': '@1hundred1', 'subscribers': 15..."



TABLE: bronze_socialblade_daily_channel_metrics

Schema:


...

,id,create_dt,ingested_at,channel_name,subscribers,subscribers_change,views,views_change,earnings_low,earnings_high,engagement,upload_frequency,average_length,channel_id,raw_record_json,raw_record_json_parsed
0,8,2026-07-18T05:23:34.066473+00:00,2026-07-18T05:23:34.066473+00:00,AKSTAR ENG,1280000,None,263670000,None,4000,None,None,None,None,@AKSTARENG,"{""channel_name"": ""AKSTAR ENG"", ""subscribers"": ...","{'channel_name': 'AKSTAR ENG', 'subscribers': ..."
1,7,2026-07-18T05:23:32.532688+00:00,2026-07-18T05:23:32.532688+00:00,8-BitRyan,4390000,None,2040000000,None,32000,None,None,None,None,@8BitRyan,"{""channel_name"": ""8-BitRyan"", ""subscribers"": 4...","{'channel_name': '8-BitRyan', 'subscribers': 4..."
2,6,2026-07-18T05:19:19.887435+00:00,2026-07-18T05:19:19.887435+00:00,AMP,8010000,None,1300000000,None,38000,None,None,None,None,@AMPEXCLUSIVE,"{""channel_name"": ""AMP"", ""subscribers"": 8010000...","{'channel_name': 'AMP', 'subscribers': 8010000..."
3,4,2026-07-18T05:19:16.544856+00:00,2026-07-18T05:19:16.544856+00:00,America's Got Talent,31400000,None,5800000000,None,—,None,None,None,None,@AGT,"{""channel_name"": ""America's Got Talent"", ""subs...","{'channel_name': 'America's Got Talent', 'subs..."
4,2,2026-07-18T05:19:13.004991+00:00,2026-07-18T05:19:13.004991+00:00,Hundred,153000,None,58940000,None,4000,None,None,None,None,@1hundred1,"{""channel_name"": ""Hundred"", ""subscribers"": 153...","{'channel_name': 'Hundred', 'subscribers': 153..."
5,1,2026-07-15T07:10:38.905933+00:00,2026-07-15T07:10:38.905933+00:00,ABC News,19700000,None,18570000000,None,528000,None,None,None,None,@ABCNews,"{""channel_name"": ""ABC News"", ""subscribers"": 19...","{'channel_name': 'ABC News', 'subscribers': 19..."



TABLE: bronze_youtube_skimmed

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,create_dt,TEXT,1,None,0
2,2,ingested_at,TEXT,1,None,0
3,3,source_file,TEXT,1,None,0
4,4,video_name,TEXT,0,None,0
5,5,channel_display_name,TEXT,0,None,0
6,6,views,TEXT,0,None,0
7,7,age,TEXT,0,None,0
8,8,channel_id,TEXT,0,None,0
9,9,raw_record_json,TEXT,1,None,0



Latest rows:


,id,create_dt,ingested_at,source_file,video_name,channel_display_name,views,age,channel_id,raw_record_json,raw_record_json_parsed
0,1399,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,TWIN LORRETTA'S DIARY is live!,TWIN LORRETTA'S DIARY,283 views,Streamed 2 days ago,@TwinLorretta,"{""age"": ""Streamed 2 days ago"", ""channel_displa...","{'age': 'Streamed 2 days ago', 'channel_displa..."
1,1398,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,The TRUTH About How Terrible Ellen DeGeneres is,Megyn Kelly,110K views,5 hours ago,@MegynKelly,"{""age"": ""5 hours ago"", ""channel_display_name"":...","{'age': '5 hours ago', 'channel_display_name':..."
2,1397,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,My House Got Volunteered | Pretty Huge Dilemma...,Pretty huge Dilemma Podcast,27 views,4 days ago,@prettyhugedilemmapodcast,"{""age"": ""4 days ago"", ""channel_display_name"": ...","{'age': '4 days ago', 'channel_display_name': ..."
3,1396,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,क से कबूतर | Ka Se Kabootar | Hindi Alphabet f...,Golu kids TV,252 views,3 days ago,@shivkumariYadav-e6g6x,"{""age"": ""3 days ago"", ""channel_display_name"": ...","{'age': '3 days ago', 'channel_display_name': ..."
4,1395,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,Prince Harry and Meghan Markle’s Royal Visit R...,The Nerve with Maureen Callahan,71K views,8 hours ago,@TheNerveShow,"{""age"": ""8 hours ago"", ""channel_display_name"":...","{'age': '8 hours ago', 'channel_display_name':..."
5,1394,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,Bermain Blood Strike,KAYSI,27 views,2 days ago,@kaysii_b,"{""age"": ""2 days ago"", ""channel_display_name"": ...","{'age': '2 days ago', 'channel_display_name': ..."
6,1393,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,FORTNITE LIVE *FREE* SPRITES GIVEAWAY #fortnit...,Mr.Rkdevil,129 views,Streamed 6 days ago,@Rk40A,"{""age"": ""Streamed 6 days ago"", ""channel_displa...","{'age': 'Streamed 6 days ago', 'channel_displa..."
7,1392,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,Congested Right VS Left sided Heart Failure an...,EasyNursingPrep,66 views,5 days ago,@EasyNursingPrep,"{""age"": ""5 days ago"", ""channel_display_name"": ...","{'age': '5 days ago', 'channel_display_name': ..."
8,1391,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,Sacred 07/13/26 France vs Brazil part 5,Sagrado Futsal de Segunda,15 views,1 day ago,@sagradofutsaldesegunda,"{""age"": ""1 day ago"", ""channel_display_name"": ""...","{'age': '1 day ago', 'channel_display_name': '..."
9,1390,2026-07-16T04:56:47.746220+00:00,2026-07-16T04:56:47.746220+00:00,https://www.youtube.com,Manvifoodofficial,Manvi food official,113 views,Streamed 7 days ago,@manvikids1,"{""age"": ""Streamed 7 days ago"", ""channel_displa...","{'age': 'Streamed 7 days ago', 'channel_displa..."


In [ ]:
connection.close()